In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import yaml
import zarr
import pandas as pd
import polars as pl
import numpy as np
from plotnine import *

from anngeno import AnnGeno
from scripts import get_burdens, get_correlation
from scripts import get_burdens_faster, get_correlation_faster

## Create new merged annotations

In [ ]:
config_path = './config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

anngeno_file = config.get("anngeno_file")
anno = pl.read_parquet(f"{anngeno_file}/annotations.parquet")
anno

In [ ]:
new_anno = pl.read_parquet('/s/project/deeprvat/ukb_gym/new_annotations/33traits_genebass_assoc/absplice2_all_varswithgenes.parquet')
new_anno

## Burdens Speedup - GPU

In [ ]:
config_path = './config_genebass.yaml'

zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/temp.zarr'

In [ ]:
get_burdens.compute_and_store_burdens(
    config_path=config_path,
    output_zarr=zarr_burdens_path,
    max_burden=True,
    only_snps=False,
    overwrite=True,
    # debug=True,
)

In [ ]:
root = zarr.group(zarr_burdens_path)
root['sum_burdens'][:, 10]

In [ ]:
root = zarr.group(zarr_burdens_path)
(np.isnan(root['max_burdens'][:, 10])==False).sum()

## Correlations Speedup

In [ ]:
config_path = './config_genebass.yaml'
zarr_file_path = '/s/project/deeprvat/ukb_gym/burdens/genebass_assocs_33_phenos.zarr'

rho_df = get_correlation.compute_correlations(config_path, zarr_file_path, max_burden=False)
rho_df

In [ ]:
rank_corr_df = rho_df.to_pandas()
config_path = './config.yaml'  # Or wherever your config file is

with open(config_path) as f:
    config = yaml.safe_load(f)

rare_variant_annotations_dict = config.get('rare_variant_annotations')

# Create a mapping from annotation name to category
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category_name, annotations in rare_variant_annotations_dict.items():
        for annotation in annotations:
            annotation_category_map[annotation] = category_name

# Add a 'category' column to rank_corr_df based on the mapping
rank_corr_df['category'] = rank_corr_df['annotation'].map(annotation_category_map)
rank_corr_df['category'] = pd.Categorical(rank_corr_df['category'], categories=['plof', 'missense', 'splicing', 'regulatory', 'misc'], ordered=True)

# Fill NaN values in 'correlation' with 0
rank_corr_df['correlation'] = rank_corr_df['correlation'].fillna(0)
rank_corr_df['abs_correlation'] = np.abs(rank_corr_df['correlation'])

rank_corr_df

In [ ]:
rank_corr_sum = rank_corr_df.query("aggregation == 'sum'").copy()
rank_corr_sum['median_abs_correlation'] = rank_corr_sum.groupby('annotation')['abs_correlation'].transform('median')
rank_corr_sum = rank_corr_sum.sort_values('median_abs_correlation', ascending=False)
rank_corr_sum['annotation'] = pd.Categorical(rank_corr_sum['annotation'], categories=rank_corr_sum['annotation'].unique(), ordered=True)

# rank_corr_df = rank_corr_df[~rank_corr_df['category'].isna()]
(
    ggplot(rank_corr_sum, aes(x='annotation', y='abs_correlation', fill='category')) +
    geom_boxplot(alpha=0.75) +
    # stat_summary(fun_data='count', geom='text', color='black', size=7, position=position_dodge(width=0.75)) +
    theme_bw() +
    scale_y_sqrt() +
    ylab('|rank correlation|') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 8),
    )
)

## Pheno vs GIS plot

In [ ]:
import zarr
from scripts import get_correlation

def pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_burdens_path):

    with open(config_path) as f:
        config = yaml.safe_load(f)

    anngeno_file = config.get('anngeno_file')
    # annotation_list = config.get('rare_variant_annotations')
    # phenotypes = config.get('phenotypes_for_association_testing')
    covs = config.get('covariates')
    prs_pheno_map_file = config.get('prs_pheno_map_file')
    prs_file = config.get('prs_file')

    cov_pheno_df = pd.read_parquet(f"{anngeno_file}/phenotypes.parquet", columns=['sample'] + covs + [phenotype]).set_index('sample')
    prs_df = pd.read_parquet(prs_file)
    prs_df = prs_df[prs_df.index.isin(cov_pheno_df.index)]
    prs_pheno_map = pd.read_csv(prs_pheno_map_file)
    prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
    all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
    pheno_corrected_df = get_correlation.cov_prs_correction(all_df, [phenotype], covs, prs_pheno_map)

    zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
    sample_list = zarr_group['samples'][:]
    gene_list = zarr_group["genes"][:]
    annotation_list = zarr_group["annotations"][:]

    gene_idx = np.where(gene_list == gene_num)[0][0]
    anno_idx = np.where(annotation_list == annotation)[0][0]
    burden_type = burden_type.lower()
    if burden_type == "max":
        burdens = zarr_group["max_burdens"][:, gene_idx, anno_idx]
    elif burden_type == "sum":
        burdens = zarr_group["sum_burdens"][:, gene_idx, anno_idx]
    else:
        raise ValueError("burden_type must be either 'max' or 'sum'")

    burden_df = pd.DataFrame(burdens, index=sample_list, columns=[gene_num]).merge(pheno_corrected_df, left_index=True, right_on='sample')
    burden_df = burden_df.dropna()
    gis_mode = burden_df[gene_num].mode()[0]
    burden_df_non_zero = burden_df[burden_df[gene_num]!=gis_mode]
    correlation = burden_df[[gene_num, phenotype]].dropna().corr(method='spearman').iloc[0, 1]

    return burden_df, burden_df_non_zero, correlation

In [ ]:
phenotype = 'LDL_direct_statin_corrected' #'Urate'
gene_num = '9138' #'9231'
annotation = 'CADD_raw' #'pangolin_score'
burden_type = 'sum'

plt_df, plt_df_nz, c = pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_file_path)

(
    ggplot(plt_df, aes(x=gene_num, y=phenotype)) +
    geom_point(alpha=0.5) +
    geom_smooth(method='lm', se=False) +
    labs(x='LDLR',
         y=phenotype) +
    theme_bw() +
    labs(title = f"{annotation} - {round(c, 4)}")

)

# Old code

## Compute correlations

In [ ]:
config_path = './config_genebass.yaml'
# zarr_file_path = '/s/project/deeprvat/ukb_gym/burdens/burdens_onlysum_onlysnps_no_vars_na.zarr'
zarr_file_path = '/s/project/deeprvat/ukb_gym/burdens/genebass_assocs_33_phenos.zarr'

root = zarr.group(zarr_file_path)
zarr_annotations = root["annotations"]
zarr_annotations[:]

In [ ]:
rho_df = get_correlation.compute_correlations(config_path, zarr_file_path, max_burden=False)

rho_df

In [ ]:
rho_df.to_parquet('/s/project/deeprvat/ukb_gym/results/genebass_assocs_33_phenos_correlations.parquet', index=False)

## Plot results

In [ ]:
rank_corr_df = pd.read_parquet('/s/project/deeprvat/ukb_gym/results/genebass_assocs_33_phenos_correlations.parquet')
config_path = './config.yaml'  # Or wherever your config file is

with open(config_path) as f:
    config = yaml.safe_load(f)

rare_variant_annotations_dict = config.get('rare_variant_annotations')

# Create a mapping from annotation name to category
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category_name, annotations in rare_variant_annotations_dict.items():
        for annotation in annotations:
            annotation_category_map[annotation] = category_name

# Add a 'category' column to rank_corr_df based on the mapping
rank_corr_df['category'] = rank_corr_df['annotation'].map(annotation_category_map)
rank_corr_df['category'] = pd.Categorical(rank_corr_df['category'], categories=['plof', 'missense', 'splicing', 'regulatory', 'misc'], ordered=True)

# Fill NaN values in 'correlation' with 0
rank_corr_df['correlation'] = rank_corr_df['correlation'].fillna(0)
rank_corr_df['abs_correlation'] = np.abs(rank_corr_df['correlation'])

rank_corr_df

In [ ]:
rank_corr_sum = rank_corr_df.query("aggregation == 'sum'").copy()
rank_corr_sum['median_abs_correlation'] = rank_corr_sum.groupby('annotation')['abs_correlation'].transform('median')
rank_corr_sum = rank_corr_sum.sort_values('median_abs_correlation', ascending=False)
rank_corr_sum['annotation'] = pd.Categorical(rank_corr_sum['annotation'], categories=rank_corr_sum['annotation'].unique(), ordered=True)

# rank_corr_df = rank_corr_df[~rank_corr_df['category'].isna()]
(
    ggplot(rank_corr_sum, aes(x='annotation', y='abs_correlation', fill='category')) +
    geom_boxplot(alpha=0.75) +
    # stat_summary(fun_data='count', geom='text', color='black', size=7, position=position_dodge(width=0.75)) +
    theme_bw() +
    scale_y_sqrt() +
    ylab('|rank correlation|') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 8),
    )
)

In [ ]:
rank_corr_max = rank_corr_df.query("aggregation == 'max'").copy()
rank_corr_max['median_abs_correlation'] = rank_corr_max.groupby('annotation')['abs_correlation'].transform('median')
rank_corr_max = rank_corr_max.sort_values('median_abs_correlation', ascending=False)
rank_corr_max['annotation'] = pd.Categorical(rank_corr_max['annotation'], categories=rank_corr_max['annotation'].unique(), ordered=True)

In [ ]:
rank_corr_max_summary = rank_corr_max.groupby(['annotation'])['abs_correlation'].agg(['mean', 'std']).reset_index().dropna()
rank_corr_max_summary['ymin'] = rank_corr_max_summary['mean'] - rank_corr_max_summary['std']
rank_corr_max_summary['ymax'] = rank_corr_max_summary['mean'] + rank_corr_max_summary['std'] 

rank_corr_max_summary = rank_corr_max_summary.sort_values('mean', ascending=False)
rank_corr_max_summary['annotation'] = pd.Categorical(rank_corr_max_summary['annotation'], categories=rank_corr_max_summary['annotation'].unique(), ordered=True)

rank_corr_max_summary

In [ ]:
(
    ggplot(rank_corr_max_summary, aes(x='annotation', y='mean', fill='category')) +
    geom_bar(stat='identity') +
    geom_errorbar(aes(ymin='ymin', ymax='ymax'), width=0.2) +
    theme_bw() +
    scale_y_sqrt() +
    ylab('|rank correlation|') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 8),
    )
)


## AbSplice2 debug

In [ ]:
a2_ukbg = pd.read_parquet('/s/project/deeprvat/ukb_gym/new_annotations/absplice2_pangolin.parquet')
a2_ukbg

In [ ]:
subset_rcdf = rank_corr_df[rank_corr_df.gene.isin(a2_ukbg.region.astype(str).unique())]
subset_rcdf

In [ ]:
(
    ggplot(rank_corr_df.query("(aggregation == 'sum') & (category == 'splicing')"), aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(8, 7),
    )
)

In [ ]:
import zarr
from scripts import get_correlation

def pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_burdens_path):

    with open(config_path) as f:
        config = yaml.safe_load(f)

    anngeno_file = config.get('anngeno_file')
    # annotation_list = config.get('rare_variant_annotations')
    # phenotypes = config.get('phenotypes_for_association_testing')
    covs = config.get('covariates')
    prs_pheno_map_file = config.get('prs_pheno_map_file')
    prs_file = config.get('prs_file')

    cov_pheno_df = pd.read_parquet(f"{anngeno_file}/phenotypes.parquet", columns=['sample'] + covs + [phenotype]).set_index('sample')
    prs_df = pd.read_parquet(prs_file)
    prs_df = prs_df[prs_df.index.isin(cov_pheno_df.index)]
    prs_pheno_map = pd.read_csv(prs_pheno_map_file)
    prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
    all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
    pheno_corrected_df = get_correlation.cov_prs_correction(all_df, [phenotype], covs, prs_pheno_map)

    zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
    sample_list = zarr_group['samples'][:]
    gene_list = zarr_group["genes"][:]
    annotation_list = zarr_group["annotations"][:]

    gene_idx = np.where(gene_list == gene_num)[0][0]
    anno_idx = np.where(annotation_list == annotation)[0][0]
    burden_type = burden_type.lower()
    if burden_type == "max":
        burdens = zarr_group["max_burdens"][:, gene_idx, anno_idx]
    elif burden_type == "sum":
        burdens = zarr_group["sum_burdens"][:, gene_idx, anno_idx]
    else:
        raise ValueError("burden_type must be either 'max' or 'sum'")

    burden_df = pd.DataFrame(burdens, index=sample_list, columns=[gene_num]).merge(pheno_corrected_df, left_index=True, right_on='sample')
    burden_df = burden_df.dropna()
    gis_mode = burden_df[gene_num].mode()[0]
    burden_df_non_zero = burden_df[burden_df[gene_num]!=gis_mode]
    correlation = burden_df[[gene_num, phenotype]].dropna().corr(method='spearman').iloc[0, 1]

    return burden_df, burden_df_non_zero, correlation

In [ ]:
phenotype = 'LDL_direct_statin_corrected' #'Urate'
gene_num = '9138' #'9231'
annotation = 'CADD_raw' #'pangolin_score'
burden_type = 'sum'

plt_df, plt_df_nz, c = pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_file_path)

(
    ggplot(plt_df, aes(x=gene_num, y=phenotype)) +
    geom_point(alpha=0.5) +
    geom_smooth(method='lm', se=False) +
    labs(x='LDLR',
         y=phenotype) +
    theme_bw() +
    labs(title = f"{annotation} - {round(c, 4)}")

)